# Coffee17 — F0 seed 42, 5-fold primary
GitHub = code; Drive = evidence/checkpoint; outer test never materialized here.


In [ ]:
ARM='F0'
from google.colab import drive
drive.mount('/content/drive')

import importlib, json, os, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

BRANCH='codex/preprocessing-study-v1'
REPO=Path('/content/coffee-bean-classification')
WORK=Path('/content')
os.chdir(WORK)
if REPO.exists():
    shutil.rmtree(REPO)

subprocess.run([
    'git','clone','--depth','1','--branch',BRANCH,
    'https://github.com/ediprin/coffee-bean-classification.git',
    str(REPO)
],check=True)
subprocess.run([
    sys.executable,'-m','pip','install','-q',
    '-r',str(REPO/'requirements/preprocessing-study.txt')
],check=True)
subprocess.run([
    sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)
],check=True)
sys.path.insert(0,str(REPO/'src'))
importlib.invalidate_caches()
os.chdir(REPO)

import torch
if not torch.cuda.is_available():
    raise RuntimeError('Aktifkan GPU Colab')

from bilinear_lmmd.core.drive_project import resolve_drive_project_root
from bilinear_lmmd.core.reproducibility import sha256_file
from bilinear_lmmd.core.run_lock import exclusive_training_lock
from bilinear_lmmd.data.preparation.prepare_coffee17 import DATASET_URL
from bilinear_lmmd.data.preparation.audit_coffee17_provenance import audit_coffee17_provenance
from bilinear_lmmd.data.preparation.prepare_preprocessing_folds import prepare_preprocessing_folds
from bilinear_lmmd.data.preparation.materialize_preprocessing_development import materialize_preprocessing_development
from bilinear_lmmd.experiments.preprocessing_environment import freeze_environment, verify_environment
from bilinear_lmmd.experiments.run_preprocessing_static_preflight import run_preprocessing_static_preflight
from bilinear_lmmd.experiments.run_preprocessing_observability_audit import run_observability_audit
from bilinear_lmmd.experiments.verify_preprocessing_reference_equivalence import verify_reference_equivalence

PROJECT=resolve_drive_project_root()
COMMIT=subprocess.check_output(
    ['git','rev-parse','HEAD'],cwd=REPO,text=True
).strip()

ARCHIVE=WORK/'coffee17_original.zip'
if not ARCHIVE.is_file():
    request=urllib.request.Request(DATASET_URL,headers={'User-Agent':'Mozilla/5.0'})
    with urllib.request.urlopen(request) as response, ARCHIVE.open('wb') as output:
        shutil.copyfileobj(response,output)

PROV_LOCAL=WORK/'coffee17_provenance'
CANONICAL=WORK/'coffee17_original_v1'
if PROV_LOCAL.exists():
    shutil.rmtree(PROV_LOCAL)
if CANONICAL.exists():
    shutil.rmtree(CANONICAL)

provenance=audit_coffee17_provenance(
    ARCHIVE,PROV_LOCAL,canonical_root=CANONICAL
)
if provenance['decision']!='PASS':
    raise RuntimeError(f"Provenance gagal: {provenance['decision']}")

FOLDS_LOCAL=WORK/'coffee17_folds'
if FOLDS_LOCAL.exists():
    shutil.rmtree(FOLDS_LOCAL)
fold_summary=prepare_preprocessing_folds(
    CANONICAL,
    PROV_LOCAL/'coffee17_provenance.json',
    FOLDS_LOCAL,
    folds=5,
    seed=42,
    validation_ratio=0.10,
)
if fold_summary['decision']!='PASS_COFFEE17_PREPROCESSING_DATA_GATE':
    raise RuntimeError('Data gate gagal')

DATA_EVIDENCE=PROJECT/'evidence/coffee17-preprocessing-data-v1'
RUNTIME_EVIDENCE=PROJECT/'evidence/coffee17-preprocessing-runtime-v1'
STATIC_EVIDENCE=PROJECT/'evidence/coffee17-preprocessing-static-v2'
OBS_EVIDENCE=PROJECT/'evidence/coffee17-preprocessing-observability-v1'
SETUP_LOCKS=PROJECT/'locks/coffee17-preprocessing-v1'
for directory in (DATA_EVIDENCE,RUNTIME_EVIDENCE,STATIC_EVIDENCE,OBS_EVIDENCE,SETUP_LOCKS):
    directory.mkdir(parents=True,exist_ok=True)

def persist_exact(source,target):
    if target.is_file():
        if sha256_file(source)!=sha256_file(target):
            raise RuntimeError(f'Evidence Drive berbeda: {target}')
    else:
        shutil.copy2(source,target)

while True:
    try:
        with exclusive_training_lock(
            SETUP_LOCKS,
            lock_name='common_setup.lock',
            stale_seconds=900,
        ):
            persist_exact(
                PROV_LOCAL/'coffee17_provenance.json',
                DATA_EVIDENCE/'coffee17_provenance.json'
            )
            persist_exact(
                PROV_LOCAL/'coffee17_raw_manifest.json',
                DATA_EVIDENCE/'coffee17_raw_manifest.json'
            )
            for name in ('clean_manifest.json','fold_manifest.json','fold_summary.json'):
                persist_exact(FOLDS_LOCAL/name,DATA_EVIDENCE/name)

            ENV=RUNTIME_EVIDENCE/'runtime_environment.json'
            LOCK=RUNTIME_EVIDENCE/'requirements_preprocessing_study_lock.txt'
            if ENV.is_file():
                verify_environment(ENV)
            else:
                freeze_environment(ENV,LOCK,authorize_freeze=True)

            STATIC=STATIC_EVIDENCE/'static_preflight.json'
            if STATIC.is_file():
                existing=json.loads(STATIC.read_text())
                if existing.get('decision')!='PASS_PREPROCESSING_STATIC_CONTRACT':
                    raise RuntimeError('Static evidence Drive belum PASS')
            else:
                LOCAL_STATIC=WORK/'static_preflight.json'
                run_preprocessing_static_preflight(
                    LOCAL_STATIC,seed=42,probe_size=64
                )
                persist_exact(LOCAL_STATIC,STATIC)

            OBS=OBS_EVIDENCE/'preprocessing_observability.json'
            if OBS.is_file():
                obs_existing=json.loads(OBS.read_text())
                if obs_existing.get('decision')!='PASS_PREPROCESSING_OBSERVABILITY_AUDIT':
                    raise RuntimeError('Observability evidence Drive belum PASS')
            else:
                LOCAL_OBS=WORK/'preprocessing_observability'
                if LOCAL_OBS.exists():
                    shutil.rmtree(LOCAL_OBS)
                run_observability_audit(
                    CANONICAL,
                    PROV_LOCAL/'coffee17_raw_manifest.json',
                    LOCAL_OBS,
                    device_name='cuda:0',
                    image_size=224,
                    expected_count=979,
                )
                persist_exact(
                    LOCAL_OBS/'preprocessing_observability.json',
                    OBS,
                )
                persist_exact(
                    LOCAL_OBS/'preprocessing_observability_per_image.csv',
                    OBS_EVIDENCE/'preprocessing_observability_per_image.csv',
                )
        break
    except RuntimeError as exc:
        if 'runtime lain' not in str(exc):
            raise
        print('Common setup sedang dipakai runtime lain; tunggu 30 detik.',flush=True)
        time.sleep(30)

ENV=RUNTIME_EVIDENCE/'runtime_environment.json'
STATIC=STATIC_EVIDENCE/'static_preflight.json'
OBS=OBS_EVIDENCE/'preprocessing_observability.json'
verify_environment(ENV)

OUT=PROJECT/'experiments/coffee17-preprocessing-primary-v1'
OUT.mkdir(parents=True,exist_ok=True)

print(
    'ARM:',ARM,
    'GPU:',torch.cuda.get_device_name(0),
    'COMMIT:',COMMIT,
    'CLEAN:',fold_summary['clean_count'],
    'OUT:',OUT
)


In [ ]:
DET=Path('/content/coffee-bean-detection-reference')
if DET.exists(): shutil.rmtree(DET)
subprocess.run(['git','clone','https://github.com/ediprin/coffee-bean-detection.git',str(DET)],check=True)
REFERENCE_COMMIT='6ef389c23932e44fe4135c32d471b3008b1cbf39'
subprocess.run(['git','-C',str(DET),'checkout','--force',REFERENCE_COMMIT],check=True)
EQ_DIR=PROJECT/'evidence/coffee17-preprocessing-reference-v2'; EQ_DIR.mkdir(parents=True,exist_ok=True)
EQUIVALENCE=EQ_DIR/'F0_luminance_reference_equivalence.json'; LOCAL_EQ=WORK/'F0_luminance_reference_equivalence.json'
verify_reference_equivalence('F0',DET,LOCAL_EQ,expected_reference_commit=REFERENCE_COMMIT)
persist_exact(LOCAL_EQ,EQUIVALENCE)


In [ ]:
for FOLD in range(1,6):
    DEV=WORK/f'coffee17_dev_fold_{FOLD}'
    if DEV.exists(): shutil.rmtree(DEV)
    materialize_preprocessing_development(CANONICAL,FOLDS_LOCAL/'clean_manifest.json',FOLDS_LOCAL/'fold_manifest.json',DEV,fold=FOLD)
    CONTRACT=DEV/'development_contract.json'; LOG=OUT/f'{ARM}_fold{FOLD}_seed42_run.log'
    command=[sys.executable,'-u','-m','bilinear_lmmd.experiments.run_preprocessing_arm','--arm',ARM,'--fold',str(FOLD),'--seed','42','--data-root',str(DEV),'--development-contract',str(CONTRACT),'--static-preflight',str(STATIC),'--observability-audit',str(OBS),'--environment',str(ENV),'--output-root',str(OUT),'--required-commit',COMMIT,'--device','cuda:0','--authorize-training']
    if EQUIVALENCE is not None: command += ['--equivalence',str(EQUIVALENCE)]
    print(f'START/RESUME {ARM} fold {FOLD}/5',flush=True)
    with LOG.open('a',encoding='utf-8') as stream:
        process=subprocess.Popen(command,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        assert process.stdout is not None
        for line in process.stdout:
            print(line,end=''); stream.write(line); stream.flush()
        rc=process.wait()
    if rc: raise RuntimeError(f'{ARM} fold {FOLD} gagal: {rc}')
    result_path=OUT/'primary'/ARM/f'fold_{FOLD}'/'seed42'/'result.json'
    result=json.loads(result_path.read_text())
    print(f"{ARM} fold {FOLD}: Macro-F1={result['metrics']['macro_f1']:.4f} | Worst-F1={result['metrics']['worst_class_f1']:.4f}")
print(f'{ARM}: 5 fold selesai. Test belum dibuka.')
